# 12-bob. 12-BOB. XAVFSIZLIK OPERATSIYALARI MARKAZIDA SI
## Kengaytirilgan amaliy ish: ogohlantirishlarni hodisalarga birlashtirish

[![Colab'da ochish](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/mrmsi/si-kiberxavfsizlik/blob/main/12-bob/12-3_ogohlantirishlarni-hodisalarga-birlashtirish.ipynb)

*"Sun'iy intellektni kiberxavfsizlikda qo'llash" darsligi, 12-bob.*

> Bu kengaytirilgan amaliy ish darslikning **60610700 – Sun'iy intellekt yo'nalishi versiyasida** keltirilgan.

Quyidagi dastur bir kunlik ogohlantirishlar oqimini modellashtiradi: oltita haqiqiy hodisa (har biri 15–60 ta ogohlantirish, bir yoki bir necha xost, umumiy foydalanuvchi va IP manzil bilan) va 250 ta bir-biriga bog'liq bo'lmagan alohida ogohlantirish. Ogohlantirishlar umumiy xost, foydalanuvchi yoki IP manzil va 30 daqiqalik vaqt oynasi bo'yicha birlashtiriladi.

**Ishga tushirish:** yuqoridagi menyudan *Runtime → Run all* (*Ishga tushirish → Hammasini bajarish*) ni tanlang yoki kataklarni birma-bir Shift+Enter bilan bajaring. Barcha ma'lumotlar dasturning o'zida yaratiladi, qo'shimcha fayl yuklash shart emas.

In [1]:
import numpy as np
from collections import Counter, defaultdict

rng = np.random.default_rng(12)

In [2]:
# 1. Bir kunlik ogohlantirishlar: 6 ta haqiqiy hodisa (har biri bir nechta ogohlantirish) + fon shovqin
ogohlantirishlar = []        # (vaqt_daqiqa, xost, foydalanuvchi, ip, hodisa_raqami)
for h in range(6):
    t0 = rng.integers(0, 1300)
    xostlar = [f"pc-{rng.integers(1, 400)}" for _ in range(rng.integers(1, 4))]
    foyd, ip = f"user{rng.integers(1, 300)}", f"203.0.113.{rng.integers(1, 255)}"
    for _ in range(rng.integers(15, 60)):               # hujum davomida turli qoidalar ishga tushadi
        ogohlantirishlar.append((t0 + rng.integers(0, 90), rng.choice(xostlar),
                                 foyd if rng.random() < 0.7 else None,
                                 ip if rng.random() < 0.5 else None, h))
for _ in range(250):                                       # bog'liq bo'lmagan alohida ogohlantirishlar
    ogohlantirishlar.append((rng.integers(0, 1440), f"pc-{rng.integers(1, 400)}",
                             f"user{rng.integers(1, 300)}" if rng.random() < 0.5 else None, None, -1))
ogohlantirishlar.sort(key=lambda o: o[0])
n = len(ogohlantirishlar)

In [3]:
# 2. Birlashtirish: umumiy obyekt (xost, foydalanuvchi, IP) va 30 daqiqalik oyna -> bitta hodisa
ota = list(range(n))
def ildiz(i):
    while ota[i] != i:
        ota[i] = ota[ota[i]]; i = ota[i]
    return i
def birlashtir(i, j):
    ota[ildiz(i)] = ildiz(j)

OYNA = 30
oxirgi = {}                                                # obyekt -> (oxirgi vaqt, ogohlantirish raqami)
for i, (vaqt, xost, foyd, ip, _) in enumerate(ogohlantirishlar):
    for obyekt in (xost, foyd, ip):
        if obyekt is None:
            continue
        if obyekt in oxirgi and vaqt - oxirgi[obyekt][0] <= OYNA:
            birlashtir(i, oxirgi[obyekt][1])
        oxirgi[obyekt] = (vaqt, i)

guruhlar = defaultdict(list)
for i in range(n):
    guruhlar[ildiz(i)].append(ogohlantirishlar[i][4])

In [4]:
# 3. Natijani baholash
haqiqiy = sum(1 for o in ogohlantirishlar if o[4] >= 0)
katta = [g for g in guruhlar.values() if len(g) >= 5]
print(f"Ogohlantirishlar: {n} ta (shundan haqiqiy hodisalarga tegishli: {haqiqiy})")
print(f"Birlashtirilgandan keyin guruhlar: {len(guruhlar)} ta, shundan 5 va undan ko'p ogohlantirishli: {len(katta)} ta")
for g in sorted(katta, key=len, reverse=True):
    asosiy, soni = Counter(g).most_common(1)[0]
    nom = f"hodisa {asosiy}" if asosiy >= 0 else "fon"
    print(f"  {len(g):>3} ta ogohlantirish, asosiy manba: {nom:<9} tozalik: {soni / len(g):.0%}")

Ogohlantirishlar: 493 ta (shundan haqiqiy hodisalarga tegishli: 243)
Birlashtirilgandan keyin guruhlar: 250 ta, shundan 5 va undan ko'p ogohlantirishli: 6 ta
   58 ta ogohlantirish, asosiy manba: hodisa 1  tozalik: 100%
   56 ta ogohlantirish, asosiy manba: hodisa 2  tozalik: 100%
   48 ta ogohlantirish, asosiy manba: hodisa 4  tozalik: 100%
   38 ta ogohlantirish, asosiy manba: hodisa 3  tozalik: 100%
   23 ta ogohlantirish, asosiy manba: hodisa 0  tozalik: 100%
   20 ta ogohlantirish, asosiy manba: hodisa 5  tozalik: 100%


## Kitobdagi natija

Dasturni ishga tushirganingizda quyidagiga o'xshash natija chiqishi kerak (tasodifiy sonlar generatori urug'i qat'iy belgilangani uchun natija odatda aynan bir xil bo'ladi; faqat bajarilish vaqti va har safar yangi yaratiladigan tasodifiy kalitlar farq qilishi mumkin):

```
Ogohlantirishlar: 493 ta (shundan haqiqiy hodisalarga tegishli: 243)
Birlashtirilgandan keyin guruhlar: 250 ta, shundan 5 va undan ko'p ogohlantirishli: 6 ta
   58 ta ogohlantirish, asosiy manba: hodisa 1  tozalik: 100%
   56 ta ogohlantirish, asosiy manba: hodisa 2  tozalik: 100%
   48 ta ogohlantirish, asosiy manba: hodisa 4  tozalik: 100%
   38 ta ogohlantirish, asosiy manba: hodisa 3  tozalik: 100%
   23 ta ogohlantirish, asosiy manba: hodisa 0  tozalik: 100%
   20 ta ogohlantirish, asosiy manba: hodisa 5  tozalik: 100%
```

## O'zingiz sinab ko'ring

Parametrlarni (ma'lumotlar hajmi, chegara qiymati, tasodifiy son urug'i) o'zgartirib, natija qanday o'zgarishini kuzating. Kitobdagi 12-bobning B darajadagi topshiriqlari aynan shu daftar asosida bajariladi.